# Loading Packages

In [1]:
import numpy as np
import pandas as pd
import glob
import os

In [2]:
def load_and_stack_xlsx(reports_path):
    # get all xlsx files
    files = glob.glob(os.path.join(reports_path, "*.xlsx"))
    
    # read and store dfs
    dfs = []
    for f in files:
        df = pd.read_excel(f)
        df["source_file"] = os.path.basename(f)  # optional but useful
        dfs.append(df)
    
    # stack (row-wise)
    stacked_df = pd.concat(dfs, ignore_index=True)
    
    return stacked_df

# Base Files

In [3]:
wrong_edit = pd.read_pickle("/home/sgw3fy/jobs/standard_shortcut/projects/model_editing/datasets/reasonvqa/rvqa_wrong_edit_df.pkl")

In [4]:
wrong_edit.rename(columns={"answer": "wrong_edit_label"}, inplace=True)

# Model: Qwen3-4B

In [5]:
reports_path = "/home/sgw3fy/jobs/standard_shortcut/projects/model_editing/repos/instruct_vlm_edit/indep_runs/temp_reports/qwen3_4b"

In [6]:
def prediction_reports(input_path, wrong_edit):
    reports_df = load_and_stack_xlsx(input_path)
    reports_df.rename(columns={"qa_pair": "qa_id"}, inplace=True)
    print(f"length df: {len(reports_df)}")

    reports_df_correct_pred = reports_df[reports_df["target"] == reports_df["pred_no_edit"]]
    reports_df_incorrect_pred = reports_df[reports_df["target"] != reports_df["pred_no_edit"]]

    # incorrect predictions analysis
    reports_df_incorrect_pred = reports_df_incorrect_pred.merge(wrong_edit[["qa_id", "wrong_edit_label"]], on="qa_id")
    reports_df_incorrect_pred["acc_wrong_edit"] = reports_df_incorrect_pred["pred_wrong_edit"] == reports_df_incorrect_pred["target"]
    reports_df_incorrect_pred["acc_correct_edit"] = reports_df_incorrect_pred["pred_correct_edit"] == reports_df_incorrect_pred["target"]
    reports_df_incorrect_pred["wrong_edit_same_no_edit_pred"] = reports_df_incorrect_pred["wrong_edit_label"] == reports_df_incorrect_pred["pred_no_edit"]

    # wrong edit, incorrect generation, correct prediction
    acc_wrong_edit_incorrect_pred = reports_df_incorrect_pred[reports_df_incorrect_pred["acc_wrong_edit"] == True]
    acc_wrong_edit_incorrect_pred["gen_wrong_edit_parsed"] = acc_wrong_edit_incorrect_pred["gen_wrong_edit"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())
    acc_wrong_edit_incorrect_pred["wrong_edit_label_parsed"] = acc_wrong_edit_incorrect_pred["wrong_edit_label"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())
    weird_cases = acc_wrong_edit_incorrect_pred[acc_wrong_edit_incorrect_pred["gen_wrong_edit_parsed"] == acc_wrong_edit_incorrect_pred["wrong_edit_label_parsed"]]    
    
    # correct predictions analysis
    reports_df_correct_pred = reports_df_correct_pred.merge(wrong_edit[["qa_id", "wrong_edit_label"]], on="qa_id")
    reports_df_correct_pred["acc_wrong_edit"] = reports_df_correct_pred["pred_wrong_edit"] == reports_df_correct_pred["target"]

    return reports_df_incorrect_pred, reports_df_correct_pred, weird_cases


In [7]:
reports_df_incorrect_pred, reports_df_correct_pred, weird_cases = prediction_reports(reports_path, wrong_edit)

length df: 1000


/tmp/ipykernel_53782/3667425221.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["gen_wrong_edit_parsed"] = acc_wrong_edit_incorrect_pred["gen_wrong_edit"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())
/tmp/ipykernel_53782/3667425221.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["wrong_edit_label_parsed"] = acc_wrong_edit_incorrect_pred["wrong_edit_label"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())


In [16]:
reports_df_incorrect_pred[reports_df_incorrect_pred["acc_wrong_edit"] == 1]

,target,pred_no_edit,gen_no_edit,pred_wrong_edit,gen_wrong_edit,pred_correct_edit,gen_correct_edit,qa_id,source_file,wrong_edit_label,acc_wrong_edit,acc_correct_edit,wrong_edit_same_no_edit_pred
1,capital punishment,torture,"Actually, there is no patron saint associated ...",capital punishment,The cause of death of this church’s patron saint,capital punishment,The cause of death of this church’s patron saint,2312,report_2312.xlsx,suicide,True,True,False
4,1623,1618,The inventor of the motor vehicle — widely cre...,1623,"The inventor of the motor vehicle, Karl Benz,",1623,The inventor of the motor vehicle is typically...,225288,report_225288.xlsx,1613,True,True,False
11,242495,219850,The castle in the image is **Château,242495,The area of the country where this castle is l...,242495,The area of the country where this castle is l...,402714,report_402714.xlsx,275537,True,True,False
19,archangel michael,crucifix,The castle in the image is **Schloss H,archangel michael,The patron saint of the country where this cas...,archangel michael,The patron saint of the country where this cas...,173984,report_173984.xlsx,john xxiii,True,True,False
22,603550,333743,The castle in the image is **Kolín,603550,590914,603550,603550,867,report_867.xlsx,590914,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,europe,south america,Lake Kärrtorp is located in **,europe,The lake shown in the image is **Lake T,europe,Europe,8122,report_8122.xlsx,australia,True,True,False
444,austria,romania,The church building shown is **St. Georg**,austria,Slovenia,austria,Austria,175018,report_175018.xlsx,slovenia,True,True,False
451,312683,301632,The palace shown in the image is **Schloss,312683,The area of the country where this palace is l...,312683,The area of the country where this palace is l...,1124,report_1124.xlsx,260402,True,True,False
454,505990,276936,The church building shown is **Sant Joan de,505990,The area of the realm where this church buildi...,505990,The area of the realm where this church buildi...,228398,report_228398.xlsx,699717,True,True,False


In [19]:
reports_df_incorrect_pred[reports_df_incorrect_pred["acc_wrong_edit"] == 1][["target", "pred_no_edit", "pred_wrong_edit", "gen_no_edit", "gen_wrong_edit", "wrong_edit_label"]].iloc[10:20]

,target,pred_no_edit,pred_wrong_edit,gen_no_edit,gen_wrong_edit,wrong_edit_label
36,united kingdom,macedonia,united kingdom,"This church, St. Mary’s Church, is",The church shown in the image is located in **,congo
39,505990,255292,505990,The castle in the image is **Castell de,671904,671904
45,europe,south america,europe,Kolbács Park is located in **,The park shown in the image is **Kol,africa
52,europe,south america,europe,This cemetery is located in **Europe**.The,The cemetery is located in **Europe**.The,asia
53,archangel michael,saint peter,archangel michael,The church building in the image is **St.,The patron saint of the sovereign state where ...,thomas aquinas
54,397,400,397,The church in question is **St. Georgs,The person for whom this church is named died in,404
74,643801,740886,643801,The château shown is **Château,The area of the country where this château,760300
87,united states,mali,united states,This is the **St. John’s Church**,"Based on the image provided, the church buildi...",macedonia
89,europe,south america,europe,This architectural complex — the **Björk,The architectural complex shown — a wooden bel...,australia
93,archangel michael,crucifix,archangel michael,The church in the image is **St. Mari,The patron saint of the country in which this ...,saint peter


In [8]:
reports_df_incorrect_pred["acc_wrong_edit"].value_counts()

acc_wrong_edit
False    364
True      98
Name: count, dtype: int64

In [12]:
reports_df_incorrect_pred["acc_correct_edit"].value_counts()

acc_correct_edit
True     457
False      5
Name: count, dtype: int64

In [10]:
len(weird_cases)

8

In [11]:
weird_cases[["pred_wrong_edit", "gen_wrong_edit", "wrong_edit_label"]]

,pred_wrong_edit,gen_wrong_edit,wrong_edit_label
22,603550,590914,590914
39,505990,671904,671904
114,643801,582731,582731
141,302068,295482,295482
169,603550,544933,544933
243,78866,68455,68455
369,9596961,8784687,8784687
392,603550,329868,329868


# Model: Qwen3-8B

In [12]:
reports_path = "/home/sgw3fy/jobs/standard_shortcut/projects/model_editing/repos/instruct_vlm_edit/indep_runs/temp_reports/qwen3_8b"

In [13]:
reports_df_incorrect_pred, reports_df_correct_pred, weird_cases = prediction_reports(reports_path, wrong_edit)

length df: 1000


/tmp/ipykernel_295716/3667425221.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["gen_wrong_edit_parsed"] = acc_wrong_edit_incorrect_pred["gen_wrong_edit"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())
/tmp/ipykernel_295716/3667425221.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["wrong_edit_label_parsed"] = acc_wrong_edit_incorrect_pred["wrong_edit_label"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())


In [14]:
reports_df_incorrect_pred["acc_wrong_edit"].value_counts()

acc_wrong_edit
False    357
True      79
Name: count, dtype: int64

In [15]:
reports_df_incorrect_pred["acc_correct_edit"].value_counts()

acc_correct_edit
True     429
False      7
Name: count, dtype: int64

In [16]:
len(weird_cases)

0

# Model: Llava

In [17]:
reports_path = "/home/sgw3fy/jobs/standard_shortcut/projects/model_editing/repos/instruct_vlm_edit/indep_runs/temp_reports/llava"

In [18]:
reports_df_incorrect_pred, reports_df_correct_pred, weird_cases = prediction_reports(reports_path, wrong_edit)

length df: 1000


/tmp/ipykernel_295716/3667425221.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["gen_wrong_edit_parsed"] = acc_wrong_edit_incorrect_pred["gen_wrong_edit"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())
/tmp/ipykernel_295716/3667425221.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["wrong_edit_label_parsed"] = acc_wrong_edit_incorrect_pred["wrong_edit_label"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())


In [19]:
reports_df_incorrect_pred["acc_wrong_edit"].value_counts()

acc_wrong_edit
False    332
True      39
Name: count, dtype: int64

In [20]:
reports_df_incorrect_pred["acc_correct_edit"].value_counts()

acc_correct_edit
True     341
False     30
Name: count, dtype: int64

In [21]:
len(weird_cases)

5

In [22]:
weird_cases[["pred_wrong_edit", "gen_wrong_edit", "wrong_edit_label"]]

,pred_wrong_edit,gen_wrong_edit,wrong_edit_label
59,312683,301781,301781
92,643801,582731,582731
144,603550,544933,544933
175,643801,458792,458792
232,10186000,9286837,9286837


# Model: Blip

In [23]:
reports_path = "/home/sgw3fy/jobs/standard_shortcut/projects/model_editing/repos/instruct_vlm_edit/indep_runs/temp_reports/blip"

In [24]:
reports_df_incorrect_pred, reports_df_correct_pred, weird_cases = prediction_reports(reports_path, wrong_edit)

length df: 1000


/tmp/ipykernel_295716/3667425221.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["gen_wrong_edit_parsed"] = acc_wrong_edit_incorrect_pred["gen_wrong_edit"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())
/tmp/ipykernel_295716/3667425221.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acc_wrong_edit_incorrect_pred["wrong_edit_label_parsed"] = acc_wrong_edit_incorrect_pred["wrong_edit_label"].apply(lambda gen_wrong_edit: str(gen_wrong_edit).strip())


In [25]:
reports_df_incorrect_pred["acc_wrong_edit"].value_counts()

acc_wrong_edit
False    256
True      40
Name: count, dtype: int64

In [26]:
reports_df_incorrect_pred["acc_correct_edit"].value_counts()

acc_correct_edit
True     286
False     10
Name: count, dtype: int64

In [27]:
len(weird_cases)

20

In [29]:
weird_cases[["pred_wrong_edit", "gen_wrong_edit", "wrong_edit_label"]]

,pred_wrong_edit,gen_wrong_edit,wrong_edit_label
18,9826675,7662912,7662912
23,romanesque architecture,stone,stone
34,italian,romanian,romanian
40,poland,ethiopia,ethiopia
41,roman empire,world war ii,world war ii
47,capital punishment,anorexia nervosa,anorexia nervosa
88,rome,south america,south america
96,german,estonian,estonian
100,gothic revival,baroque,baroque
123,romanesque architecture,underground culture,underground culture


In [6]:
all_indices = list(range(4370))

for start in range(0, len(all_indices), 20):
    end = start + 20
    if end >= len(all_indices):
        break
    print(f"{all_indices[start]}-{all_indices[end]}", end=", ")

0-20, 20-40, 40-60, 60-80, 80-100, 100-120, 120-140, 140-160, 160-180, 180-200, 200-220, 220-240, 240-260, 260-280, 280-300, 300-320, 320-340, 340-360, 360-380, 380-400, 400-420, 420-440, 440-460, 460-480, 480-500, 500-520, 520-540, 540-560, 560-580, 580-600, 600-620, 620-640, 640-660, 660-680, 680-700, 700-720, 720-740, 740-760, 760-780, 780-800, 800-820, 820-840, 840-860, 860-880, 880-900, 900-920, 920-940, 940-960, 960-980, 980-1000, 1000-1020, 1020-1040, 1040-1060, 1060-1080, 1080-1100, 1100-1120, 1120-1140, 1140-1160, 1160-1180, 1180-1200, 1200-1220, 1220-1240, 1240-1260, 1260-1280, 1280-1300, 1300-1320, 1320-1340, 1340-1360, 1360-1380, 1380-1400, 1400-1420, 1420-1440, 1440-1460, 1460-1480, 1480-1500, 1500-1520, 1520-1540, 1540-1560, 1560-1580, 1580-1600, 1600-1620, 1620-1640, 1640-1660, 1660-1680, 1680-1700, 1700-1720, 1720-1740, 1740-1760, 1760-1780, 1780-1800, 1800-1820, 1820-1840, 1840-1860, 1860-1880, 1880-1900, 1900-1920, 1920-1940, 1940-1960, 1960-1980, 1980-2000, 2000-2020